# ReefShield — Event Mining (offline)

Loads a **configurable** event from local processed files and shows the IMERG
rainfall series, rolling accumulations, wettest windows, ERA5-Land variables,
antecedent features, data completeness and the candidate table.

**Offline by design.** Every cell reads files already on disk. No credentials
are needed and no network request is made. Production logic lives in
`backend/src/`; this notebook only calls it.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))

from ingestion.imerg import wettest_windows
from processing.event_mining import rank_rainfall_candidates, separate_by_run_type

# ---- CONFIGURE THE EVENT HERE -------------------------------------------
EVENT_ID = "AQ-2016-10-28"      # change this to load a different event
# --------------------------------------------------------------------------

processed = PROJECT_ROOT / "data" / "processed" / "events" / EVENT_ID
print("event      :", EVENT_ID)
print("processed  :", processed)
print("exists     :", processed.is_dir())
for path in sorted(processed.glob("*")):
    print(f"  {path.name}  ({path.stat().st_size/1024:.1f} KB)")

## 1. Load the IMERG window

In [ ]:
imerg_path = processed / f"{EVENT_ID}_imerg.nc"
imerg = xr.open_dataset(imerg_path)

print("run type       :", imerg.attrs.get("imerg_run_type"))
print("product        :", imerg.attrs.get("source_product"))
print("training-safe  :", imerg.attrs.get("suitable_for_training"))
print("granules       :", imerg.attrs.get("granule_count"))
print("completeness   :", f'{float(imerg.attrs.get("data_completeness_percent", float("nan"))):.2f} %')
print("window         :", imerg.attrs.get("first_timestamp_utc"), "->", imerg.attrs.get("last_timestamp_utc"))
print("shape          :", imerg["precipitation"].shape)
imerg

## 2. Rainfall time series and rolling accumulations

In [ ]:
times = pd.to_datetime([t.strftime("%Y-%m-%dT%H:%M:%S") for t in np.atleast_1d(imerg["time"].values)])
rate = np.nanmax(np.asarray(imerg["precipitation"].values, dtype="float64"), axis=(1, 2))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(times, rate, lw=1.2, color="#1f77b4")
axes[0].set_ylabel("mm/hr")
axes[0].set_title(f"{EVENT_ID} — IMERG box-maximum precipitation rate")
axes[0].grid(alpha=0.3)

for name, colour in (("rain_1h_mm", "#7fbf7f"), ("rain_3h_mm", "#1f77b4"),
                     ("rain_6h_mm", "#ff7f0e"), ("rain_24h_mm", "#d62728")):
    if name in imerg.variables:
        series = np.nanmax(np.asarray(imerg[name].values, dtype="float64"), axis=(1, 2))
        axes[1].plot(times, series, lw=1.3, label=name, color=colour)
axes[1].set_ylabel("mm"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title("Trailing rolling accumulations (interval-end labelling)")
plt.tight_layout(); plt.show()

## 3. Wettest windows

In [ ]:
peaks = wettest_windows(imerg, rolling_windows_hours=imerg.attrs.get("rolling_windows_hours", [1, 3, 6, 24]))
rows = []
for name, info in peaks.items():
    if info.get("max_mm") is not None:
        rows.append({"window": name, "max_mm": round(info["max_mm"], 4),
                     "start_utc": info["window_start_utc"], "end_utc": info["window_end_utc"],
                     "lat": info["lat"], "lon": info["lon"]})
pd.DataFrame(rows)

## 4. ERA5-Land variables

Normalised hourly fields. `temporal_semantics_mode` records how the accumulated
variables were interpreted, and the evidence that proved it.

In [ ]:
era5_path = processed / f"{EVENT_ID}_era5_land.nc"
if era5_path.exists():
    era5 = xr.open_dataset(era5_path)
    print("semantics mode :", era5.attrs.get("temporal_semantics_mode"))
    print("evidence       :", era5.attrs.get("temporal_semantics_evidence"))
    print("timestamps     :", era5["time"].size)
    print("variables      :", sorted(era5.data_vars))

    e_times = pd.to_datetime(np.atleast_1d(era5["time"].values))
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    if "total_precipitation_hourly_mm" in era5:
        ax[0].bar(e_times, np.nanmax(np.asarray(era5["total_precipitation_hourly_mm"].values, dtype="float64"), axis=(1, 2)), width=0.03, color="#1f77b4")
        ax[0].set_ylabel("mm/h"); ax[0].set_title("ERA5-Land hourly precipitation (box max)"); ax[0].grid(alpha=0.3)
    if "swvl1" in era5:
        ax[1].plot(e_times, np.nanmean(np.asarray(era5["swvl1"].values, dtype="float64"), axis=(1, 2)), color="#8c564b")
        ax[1].set_ylabel("m3/m3"); ax[1].set_title("Soil moisture layer 1 (land-cell mean)"); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No ERA5-Land file for this event — skipping.")

## 5. Antecedent features

Missing hours are excluded from sums, never zero-filled. `quality_flag` records
what each cell's data actually supported.

In [ ]:
features_path = processed / f"{EVENT_ID}_antecedent_features.parquet"
if features_path.exists():
    features = pd.read_parquet(features_path)
    print("rows:", len(features))
    display(features.head(10))
    print("\nquality flags:"); print(features["quality_flag"].value_counts())
else:
    print("No antecedent features for this event — skipping.")

## 6. Data completeness and sea mask

In [ ]:
precip = np.asarray(imerg["precipitation"].values, dtype="float64")
print(f"IMERG valid cells   : {int(np.isfinite(precip).sum())}")
print(f"IMERG missing cells : {int(np.isnan(precip).sum())}")
print(f"IMERG completeness  : {float(imerg.attrs.get('data_completeness_percent', float('nan'))):.2f} %")

if era5_path.exists() and "swvl1" in era5:
    soil = np.asarray(era5["swvl1"].values, dtype="float64")
    sea = np.isnan(soil).any(axis=0)
    print(f"\nERA5 land cells     : {int((~sea).sum())}")
    print(f"ERA5 sea-mask cells : {int(sea.sum())}  (permanent, never interpolated)")

## 7. Candidate table

**`is_exhaustive` is False** — these candidates come from the configured
windows only, not a complete historical archive.

In [ ]:
candidates_path = PROJECT_ROOT / "data" / "processed" / "events" / "rainfall_candidates.parquet"
if candidates_path.exists():
    candidates = pd.read_parquet(candidates_path)
    print("rows          :", len(candidates))
    print("is_exhaustive :", bool(candidates["is_exhaustive"].iloc[0]) if len(candidates) else "n/a")
    print("scope         :", candidates["candidate_generation_scope"].iloc[0] if len(candidates) else "n/a")
    display(candidates[["event_id", "peak_time_utc", "rain_3h_mm", "rain_24h_mm",
                        "historical_percentile", "anomaly_score", "run_type",
                        "quality_score", "is_exhaustive"]])
else:
    print("Run scripts/build_rainfall_candidates.py first.")

## 8. Quality and limitations

Read before using any number above.

1. **Candidate scope is not exhaustive.** `is_exhaustive = False`. These are the
   configured demonstration windows, not every historical event.
2. **IMERG resolution.** ~11 km cells smooth the localized convective storms
   that drive Aqaba flash floods — a documented product limitation.
3. **ERA5-Land is land-only.** 3 of 20 cells are permanently NaN over the Gulf;
   coastal catchments will always show reduced valid area.
4. **Grids are not index-aligned.** ERA5-Land and IMERG cell centres are offset
   by half a cell. Combine only by area-weighted overlap.
5. **Early Run is preliminary.** `suitable_for_training = false`; values are
   revised in Late and Final runs, and granules can be missing.
6. **Antecedent windows may be partial.** A 48-hour ERA5 window cannot fill a
   168-hour lookback — flagged `PARTIAL_WINDOW`, never zero-filled.
7. **Catchment aggregation is not included** — it requires
   `catchments_PROVISIONAL.gpkg` (task P1), which does not exist yet.
8. **Derived rainfall peak vs documented flood arrival.** For `AQ-2016-10-28`
   the IMERG peak falls *after* the reported arrival; the likely cause is that
   the generating catchments lie outside this box. See `docs/event_dates.md`.


In [ ]:
imerg.close()
try:
    era5.close()
except NameError:
    pass
print("closed")